# Занятие 1. Spark DataFrame API

Работаем с небольшой выборкой данных розничной сети — той же, что будет на следующих
занятиях: один день операций касс (~38 тыс. строк) и справочники. Описание полей —
[docs/retail-data.md](../docs/retail-data.md).

План:

0. Проверить, что стенд работает
1. DataFrame = строки + схема
2. Чтение CSV из HDFS
3. Ленивые вычисления: transformations и actions
4. Основные операции: `select` / `withColumn`, `filter`, `sort`, `groupBy`, `join`, `toPandas`
5. План запроса: `explain()`
6. Партиции, задачи и стадии
7. Результат — в таблицу Iceberg
8. Задания

**Перед началом** положите выборку в HDFS — из корня репозитория, в терминале:

```bash
docker compose exec namenode hdfs dfs -mkdir -p /raw
docker compose exec namenode hdfs dfs -put -f /data/sample /raw/
docker compose exec namenode hdfs dfs -ls /raw/sample
```

Каталог `./data` репозитория виден в контейнере namenode как `/data`. Зачем HDFS: файлы
читают executor'ы в контейнерах, диска вашей машины они не видят.

In [ ]:
import sys
import time
sys.path.append("../host")
from spark_session import get_spark

import pandas as pd
from pyspark.sql import Row, functions as F

spark = get_spark("lab-01")

## 0. Стенд работает

Таблица Iceberg в HDFS: создать, записать строку, прочитать. Если ячейка выполнилась —
кластер, каталог и хранилище на месте.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.demo")
spark.sql("CREATE TABLE IF NOT EXISTS iceberg.demo.t (id BIGINT, name STRING) USING iceberg")
spark.sql("INSERT INTO iceberg.demo.t VALUES (1, 'a')")
# Каждый запуск ячейки добавляет ещё одну строку (1, 'a'): INSERT не идемпотентен.
# Как делать загрузки, которые можно безопасно повторять, — тема занятия 2.
spark.table("iceberg.demo.t").show()

## 1. DataFrame = строки + схема

DataFrame — таблица: набор строк и **схема**, то есть имена и типы колонок. Создать его
можно из списка Python, из pandas, из файлов, из таблицы или из базы данных по JDBC.

In [ ]:
stores_small = spark.createDataFrame([
    Row(store_id=1, city="Пермь",        format="супермаркет", area_m2=850.0),
    Row(store_id=2, city="Екатеринбург", format="гипермаркет", area_m2=4200.0),
    Row(store_id=3, city="Пермь",        format="у дома",      area_m2=180.0),
])
stores_small.printSchema()
stores_small.show()

In [ ]:
# Из pandas: типы колонок Spark выводит из типов pandas
pdf = pd.DataFrame({"store_id": [1, 2, 3], "plan_rub": [1.5e6, 7.0e6, 0.4e6]})
plan = spark.createDataFrame(pdf)
plan.printSchema()
plan.show()

## 2. Чтение CSV из HDFS

Что происходит в момент `spark.read.csv(...)`:

1. Spark получает **список файлов** по пути. Нет файла или прав — ошибка сразу.
2. Spark определяет **схему**: берёт её из заголовка и из `schema=...` или, с
   `inferSchema=True`, **читает весь файл**, чтобы угадать типы. Это отдельное задание
   на кластере — посмотрите на http://localhost:4040 → *Jobs*.

Сами данные не читаются: DataFrame — это ссылка на файлы плюс схема.

In [ ]:
RAW = "hdfs://namenode:9000/raw/sample"

sales_guess = spark.read.csv(f"{RAW}/sales.csv", header=True, inferSchema=True)
sales_guess.printSchema()

Spark угадал почти всё, но `event_ts` мог стать строкой, а `customer_id` — `int`
вместо `bigint`. На больших данных угадывание ещё и дорого. Поэтому схему задают явно —
строкой в стиле SQL:

In [ ]:
SALES_SCHEMA = """
    event_id BIGINT, event_ts TIMESTAMP, receipt_id BIGINT, line_no INT,
    store_id INT, product_id INT, category_id INT, customer_id BIGINT,
    is_loyalty BOOLEAN, operation_type STRING, quantity INT,
    price_regular_kop BIGINT, price_paid_kop BIGINT, payment_method STRING,
    ref_event_id BIGINT
"""

sales = spark.read.csv(f"{RAW}/sales.csv", header=True, schema=SALES_SCHEMA,
                       timestampFormat="yyyy-MM-dd HH:mm:ss.SSS")
# Справочники маленькие — тут угадывание типов допустимо.
# escape='"': в названиях есть кавычки и запятые
stores = spark.read.csv(f"{RAW}/stores.csv", header=True, inferSchema=True, escape='"')
categories = spark.read.csv(f"{RAW}/categories.csv", header=True, inferSchema=True, escape='"')

sales.show(5, truncate=False)
print("строк:", sales.count())

## 3. Ленивые вычисления

Операции над DataFrame бывают двух видов:

| вид | примеры | что происходит |
|---|---|---|
| **transformation** | `select`, `withColumn`, `filter`, `groupBy().agg()`, `join`, `orderBy` | возвращает **новый DataFrame** — описание вычисления. Данные не читаются |
| **action** | `show`, `count`, `collect`, `toPandas`, `write...` | Spark строит план и **выполняет** его на кластере |

pandas вычисляет каждую строчку сразу и хранит промежуточные таблицы в памяти. Spark
только копит цепочку преобразований, а в момент action оптимизирует её целиком.

In [ ]:
t0 = time.time()
revenue_by_store = (
    sales
    .filter(F.col("operation_type") == "SALE")
    .withColumn("price_rub", F.col("price_paid_kop") / 100)
    .groupBy("store_id")
    .agg(F.sum("price_rub").alias("revenue_rub"))
)
print(f"цепочка построена за {time.time() - t0:.3f} с — данные ещё не читались")

t0 = time.time()
revenue_by_store.orderBy(F.desc("revenue_rub")).show(5)
print(f"show() выполнен за {time.time() - t0:.1f} с — вот теперь Spark посчитал")

Суммы вида `262091.30000000016` — не ошибка Spark, а свойство чисел с плавающей точкой
(`double`): 0,1 в двоичной записи не представимо точно. Поэтому в источнике деньги хранятся
в **копейках целыми числами**, а в рубли переводятся в самом конце, с округлением.

### Какие ошибки появляются сразу, а какие — только при action

* **Ошибка анализа** (нет колонки, неверный тип) — сразу, при построении цепочки: Spark
  проверяет её по схеме, не читая данных.
* **Ошибка в данных** — только при action, когда до плохой строки дойдёт очередь.

In [ ]:
try:
    sales.select("no_such_column")
except Exception as e:
    print("сразу:", type(e).__name__, "-", str(e).splitlines()[0][:120])

In [ ]:
# Проверка «цена не отрицательная»: transformation проходит без ошибок...
checked = sales.withColumn("ok", F.assert_true(F.col("price_paid_kop") >= 0))
print("цепочка построена")

# ...а action натыкается на битые строки источника.
# Нужен action, который использует колонку ok: для checked.count() она не нужна,
# оптимизатор выбросит её вместе с проверкой, и ошибки не будет.
try:
    checked.agg(F.count("ok")).collect()
except Exception as e:
    print("при action:", type(e).__name__)
    print([line.strip() for line in str(e).splitlines() if "is not true" in line][:1])

## 4. Основные операции

### `select` и `withColumn` — выбрать и вычислить колонки

Функции для колонок — в модуле `pyspark.sql.functions` (у нас `F`). Встроенные функции
выполняются внутри JVM и работают быстро; свой код на Python (UDF) заметно медленнее.

In [ ]:
lines = (
    sales
    .select("event_ts", "receipt_id", "store_id", "category_id", "is_loyalty",
            "operation_type", "quantity", "price_regular_kop", "price_paid_kop", "payment_method")
    .withColumn("price_rub", F.col("price_paid_kop") / 100)
    .withColumn("discount_rub", (F.col("price_regular_kop") - F.col("price_paid_kop")) / 100)
    .withColumn("hour", F.hour("event_ts"))
)
lines.show(5)

### `filter` (он же `where`) — оставить нужные строки

In [ ]:
sold = lines.filter((F.col("operation_type") == "SALE") & (F.col("price_paid_kop") >= 0))
print("продаж:", sold.count(), "из", lines.count())

# условия пишутся через & | ~ и обязательно в скобках; пустые значения — isNull / isNotNull
print("операций без покупателя:", sales.filter(F.col("customer_id").isNull()).count())

### `orderBy` (он же `sort`) — упорядочить

In [ ]:
# Самые дорогие позиции
sold.orderBy(F.desc("price_rub")).select("receipt_id", "quantity", "price_rub", "discount_rub").show(5)

### `groupBy` + `agg` — агрегаты по группам

Как `GROUP BY` в SQL: `count`, `sum`, `avg`, `min`, `max`, `countDistinct` и другие.

In [ ]:
by_hour = (
    sold.groupBy("hour")
        .agg(F.count("*").alias("lines"),
             F.countDistinct("receipt_id").alias("receipts"),
             F.round(F.sum("price_rub"), 2).alias("revenue_rub"))
        .orderBy("hour")
)
by_hour.show(24)

### `join` — соединить со справочником

`join(другой_df, "ключ", "тип")`. Типы: `inner` (по умолчанию), `left`, `right`, `full`,
а также `left_semi` / `left_anti` — «есть пара» / «нет пары», без добавления колонок.

In [ ]:
by_region = (
    sold.join(stores, "store_id")                 # inner join по store_id
        .groupBy("region")
        .agg(F.countDistinct("receipt_id").alias("receipts"),
             F.round(F.sum("price_rub"), 2).alias("revenue_rub"))
        .withColumn("avg_check_rub", F.round(F.col("revenue_rub") / F.col("receipts"), 2))
        .orderBy(F.desc("revenue_rub"))
)
by_region.show(truncate=False)

# left_anti: операции с магазином, которого нет в справочнике
print("неизвестных магазинов:", sales.join(stores, "store_id", "left_anti").count())

### `toPandas` — забрать результат на свою машину

`toPandas()` — тоже action: собирает **все** строки в память driver'а. Для агрегатов из
десятков строк — отлично (графики, отчёт). Для сырых данных — никогда: на настоящих
объёмах driver упадёт с нехваткой памяти.

In [ ]:
pdf_region = by_region.toPandas()
pdf_region.plot.barh(x="region", y="revenue_rub", figsize=(8, 5), legend=False,
                     title="Выручка за день по регионам, руб.");

## 5. План запроса

`explain()` показывает, **как** Spark выполнит запрос. Возьмём запрос попроще —
выручка по регионам — и вызовем `explain(True)`, он печатает все этапы:

1. **Parsed Logical Plan** — ваша цепочка как есть;
2. **Analyzed Logical Plan** — колонки и типы проверены по схеме;
3. **Optimized Logical Plan** — оптимизатор Catalyst переставил и упростил операции;
4. **Physical Plan** — конкретные алгоритмы: как читать, как соединять, где shuffle.

In [ ]:
region_revenue = (
    sold.join(stores, "store_id")
        .groupBy("region")
        .agg(F.sum("price_rub").alias("revenue_rub"))
)
region_revenue.explain(True)

Физический план читается **снизу вверх**. Найдите в нём:

* `FileScan csv ... PushedFilters: [...], ReadSchema: struct<...>` — фильтр передан в
  чтение файла, и читаются только нужные колонки, хотя `filter` в коде стоит после `select`;
* `BroadcastHashJoin` + `BroadcastExchange` — справочник магазинов маленький, Spark
  рассылает его копию каждому executor'у, и большую таблицу перемешивать не нужно;
* `HashAggregate (partial)` → `Exchange hashpartitioning(region)` → `HashAggregate (final)` —
  каждая задача сначала считает частичные суммы по своему куску данных, затем строки
  одного региона пересылаются в одну задачу (**shuffle**) и суммы складываются;
* `Exchange` — граница **стадий**: до него одна стадия, после — следующая;
* `AdaptiveSparkPlan isFinalPlan=false` — план ещё не выполнялся. Во время выполнения
  адаптивный режим (AQE) может его поправить, например объединить мелкие партиции.

У `by_region` из раздела 4 план сложнее: `countDistinct` требует ещё одного shuffle —
сравните `by_region.explain()`.

Сравните с запросом, где фильтр стоит в самом конце — оптимизированный план тот же:

In [ ]:
late_filter = (
    sales.withColumn("price_rub", F.col("price_paid_kop") / 100)
         .select("store_id", "operation_type", "price_rub")
         .filter(F.col("operation_type") == "SALE")
)
late_filter.explain()

## 6. Партиции, задачи и стадии

Spark делит данные на **партиции** и обрабатывает каждую отдельной **задачей** (task):
одна партиция — одна задача. Задачи одной **стадии** выполняются параллельно на ядрах
executor'ов, у нас их 4.

CSV можно резать на куски по байтам (он **splittable**), поэтому один большой файл тоже
читается параллельно. Размер куска задаёт `spark.sql.files.maxPartitionBytes`
(по умолчанию 128 МБ). Наш файл — 3 МБ, это одна партиция:

In [ ]:
print("партиций при чтении:", sales.rdd.getNumPartitions())

spark.conf.set("spark.sql.files.maxPartitionBytes", "512k")      # режем мельче
spark.conf.set("spark.sql.adaptive.enabled", "false")           # без AQE, см. ниже
sales_split = spark.read.csv(f"{RAW}/sales.csv", header=True, schema=SALES_SCHEMA,
                             timestampFormat="yyyy-MM-dd HH:mm:ss.SSS")
print("партиций при чтении кусками по 512 КБ:", sales_split.rdd.getNumPartitions())

sales_split.groupBy("payment_method").count().show()
spark.conf.unset("spark.sql.files.maxPartitionBytes")
spark.conf.unset("spark.sql.adaptive.enabled")

Откройте http://localhost:4040 → **Jobs** → последнее задание → **DAG Visualization**:

* **стадия 1** — столько задач, сколько партиций у файла (7): чтение + частичный подсчёт;
* **стадия 2** — после shuffle, задач столько, сколько партиций после `Exchange`
  (`spark.sql.shuffle.partitions`, у нас 8).

В ячейке выше выключено **адаптивное выполнение** (AQE, `spark.sql.adaptive.enabled`),
чтобы картина была классической. С AQE, который в Spark 3.5 включён по умолчанию, Spark
перестраивает план на ходу по реальным размерам данных: запускает стадию до shuffle
отдельным заданием, а потом объединяет мелкие shuffle-партиции (здесь 8 → 1), меняет тип
join'а, дробит перекошенные партиции. Уберите строку с `adaptive.enabled`, перезапустите
ячейку и сравните: заданий станет два, во втором — одна задача, а стадия чтения будет
серой (*skipped*), потому что уже выполнена в первом.

Строка прогресса в ноутбуке `[Stage 12:=====>  (3 + 4) / 7]` читается так: в стадии 7
задач, 3 уже готовы, 4 выполняются прямо сейчас — по одной на каждое ядро.

**Узкие** преобразования (`filter`, `select`, `withColumn`) работают внутри партиции и
стадию не разрывают. **Широкие** (`groupBy`, `join` двух больших таблиц, `orderBy`,
оконные функции) требуют переслать строки между executor'ами — это shuffle и новая стадия.

## 7. Результат — в таблицу Iceberg

Запись — тоже action. Результат сохраняется в HDFS как таблица Iceberg и переживает
перезапуск ноутбука.

In [ ]:
by_region.writeTo("iceberg.demo.revenue_by_region").using("iceberg").createOrReplace()
spark.table("iceberg.demo.revenue_by_region").show(truncate=False)

## 8. Задания

Работайте с DataFrame `sales`, `stores`, `categories` из раздела 2. Учитывайте только
продажи с неотрицательной ценой — готовый DataFrame `sold` из раздела 4.

### Задание 1. Способы оплаты

Для каждого `payment_method` посчитайте число позиций, число чеков и выручку в рублях.
Отсортируйте по выручке по убыванию. Какой способ оплаты самый популярный?

In [ ]:
# Задание 1
payments = ...  # ваш код

payments.show()

### Задание 2. Выручка по группам категорий

Соедините продажи со справочником `categories` по `category_id` и посчитайте выручку по
`group_name`. Заберите результат в pandas и постройте горизонтальную столбчатую диаграмму.

In [ ]:
# Задание 2
by_group = ...  # ваш код

by_group.show(truncate=False)
# by_group.toPandas().plot.barh(x="group_name", y="revenue_rub", legend=False);

### Задание 3. Скидки у участников программы

Добавьте колонку `discount_pct` — скидка в процентах от цены без скидки:
`100 * (price_regular_kop - price_paid_kop) / price_regular_kop`. Для участников бонусной
программы и остальных (`is_loyalty`) посчитайте среднюю стоимость позиции в рублях и
среднюю скидку в процентах. У кого скидка больше?

In [ ]:
# Задание 3
loyalty = ...  # ваш код

loyalty.show()

### Задание 4. Прочитайте план

Вызовите `explain()` для результата задания 2 и ответьте:

1. Каким способом Spark соединил продажи с категориями и почему?
2. Сколько раз в плане встречается `Exchange` — то есть сколько будет стадий?
3. Какие колонки на самом деле читаются из файла продаж (`ReadSchema`)?

Проверьте ответ 2 в Spark UI: http://localhost:4040 → *Jobs*.

In [ ]:
# Задание 4

## Итоги

* DataFrame — не данные, а **описание** вычисления: ссылка на источник, схема и цепочка
  преобразований
* Transformations ничего не вычисляют, **actions** запускают выполнение
* Перед выполнением Spark оптимизирует цепочку: фильтры и выбор колонок уходят в чтение
* Партиция = задача; широкие операции (`groupBy`, `join`, `orderBy`) добавляют **shuffle**
  и новую стадию
* `toPandas()` — только для маленьких результатов

Остановите сессию: без этого кластер занят, и следующий ноутбук или DAG будут ждать.

In [ ]:
spark.stop()